# Task 6: High-Throughput Asynchronous RAG Pipeline with Cross-Encoder Reranking


## Objective

Build a compact asynchronous RAG pipeline with bi-encoder retrieval, cross-encoder reranking, and concurrent synthesis.


## Short Theory

RAG retrieves candidate context before generation. A bi-encoder is efficient for retrieval; a cross-encoder can rerank query-document pairs more precisely.


## Step 1: Imports


In [1]:
import asyncio
from sentence_transformers import SentenceTransformer, CrossEncoder


## Step 2: Retrieval + Reranking


In [2]:
documents = [
    "Transformers use self-attention.",
    "RAG retrieves external knowledge before generation.",
    "BPE builds subword vocabularies.",
    "Diffusion models learn denoising."
]
bi = SentenceTransformer("all-MiniLM-L6-v2")
cross = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "How does RAG work?"
emb = bi.encode(documents, normalize_embeddings=True)
q = bi.encode([query], normalize_embeddings=True)[0]
scores = emb @ q
candidates = [documents[i] for i in scores.argsort()[-3:][::-1]]
rerank = cross.predict([(query, d) for d in candidates])
print(list(zip(candidates, rerank)))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\obito\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\obito\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[('RAG retrieves external knowledge before generation.', np.float32(2.325218)), ('Transformers use self-attention.', np.float32(-11.314322)), ('BPE builds subword vocabularies.', np.float32(-11.402364))]


## Step 3: Async Synthesis


In [6]:
import asyncio

async def synthesize(context):
    await asyncio.sleep(0.01)
    return "Answer using: " + " | ".join(context)

result = await synthesize(candidates)
print(result)

Answer using: RAG retrieves external knowledge before generation. | Transformers use self-attention. | BPE builds subword vocabularies.


## Small Experiment

Concurrency Check


In [8]:
import asyncio

async def job(i):
    await asyncio.sleep(0.01)
    return f"request-{i} done"

results = await asyncio.gather(
    *(job(i) for i in range(3))
)

print(results)

['request-0 done', 'request-1 done', 'request-2 done']


## Conclusion

Demonstrated the dual-stage retrieval/reranking design and asynchronous request orchestration.
